<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/10_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10 — Deployment

> **Where you are** — everything so far ran inside this notebook. Deployment = the same agent, running as a server.
> - **This module is a guided tour:** you run one local server; the Docker / Cloud Run / Agent Engine cells are read-along, nothing to install.
> - **First met in M02, used again here:** a subprocess — this time *you* start it (`subprocess.Popen`), the way `McpToolset` did it for you.

The last module of Part 1. Nine modules of agents living in notebook cells — but nobody ships a notebook. A product is an **HTTP service**: something a website, an app, or a colleague's script can send requests to. This module turns your agent into exactly that.

One honest promise up front, so you can relax: **you will not deploy to any cloud today.** That needs billing accounts and patience. What you *will* do is run your agent as a real local HTTP server and talk to it — the same shape it will have in production, just on your machine.

**What we'll do:**

1. Put the agent into the folder shape every deployment tool expects.
2. Run it as a local HTTP service with `adk api_server` — and call it the way a customer would.
3. Read through the three cloud paths: plain Docker (any cloud), `adk deploy cloud_run` (one command, Google), and Agent Engine (managed, most batteries included).
4. Meet **Plugins** — one rule applied to every agent in an app at once.
5. Walk the production checklist.

**Running cost:** $0 — everything in this notebook is local.

# Setup

Same ritual — install and key. The key is for the *running agent*; deployment itself needs nothing extra.

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


In [2]:
import os, sys, warnings, tempfile, shutil, time, subprocess
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()

# API key (for the running agent, not for deployment itself)
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    except ImportError: pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ Environment ready.")

✅ Environment ready.


# The Folder Every Deploy Tool Expects

The first thing any deployment tool asks is: *"where is your agent?"* They all expect the same answer — a folder:

```
my_agent/
├── __init__.py          # empty; makes the folder a Python package
├── agent.py             # exports `root_agent`
├── requirements.txt     # pinned Python deps
└── .env                 # optional; OPENROUTER_API_KEY etc.
```

You have seen this shape before — it is the same folder `adk web` asked for in M01. One new file: `requirements.txt`, the shopping list of packages, pinned so the server installs exactly the versions you tested with. `adk api_server`, `adk web` and `adk deploy` all open `agent.py` and look for one variable named `root_agent`. That is the whole contract.

The next cell writes this layout into a temporary directory. Read the `agent.py` it creates — it is a Module-01-style weather agent, nothing more.

In [3]:
DEPLOY_DIR = tempfile.mkdtemp(prefix="adk_m10_")
AGENT_DIR = os.path.join(DEPLOY_DIR, "my_weather_agent")
os.makedirs(AGENT_DIR, exist_ok=True)

# __init__.py — make it a package
with open(os.path.join(AGENT_DIR, "__init__.py"), "w") as f:
    f.write("")

# agent.py — the thing the platforms look for
AGENT_CODE = (
    "import os\n"
    "from dotenv import load_dotenv\n"
    "load_dotenv()\n\n"
    "from google.adk.agents import LlmAgent\n"
    "from google.adk.models.lite_llm import LiteLlm\n\n"
    "def get_weather(city: str) -> dict:\n"
    "    'Look up today\u2019s weather for a city.'\n"
    "    db = {\n"
    "        'Bratislava': {'city': 'Bratislava', 'condition': 'Sunny', 'temperature_c': 18},\n"
    "        'Prague': {'city': 'Prague', 'condition': 'Cloudy', 'temperature_c': 14},\n"
    "        'Munich': {'city': 'Munich', 'condition': 'Rainy', 'temperature_c': 11},\n"
    "    }\n"
    "    return db.get(city, {'error': f'No data for {city}'})\n\n"
    "root_agent = LlmAgent(\n"
    "    name='weather_agent',\n"
    "    model=LiteLlm(model='openrouter/openai/gpt-5.6-luna'),\n"
    "    description='Reports weather for European cities.',\n"
    "    instruction='Use get_weather. Answer in one sentence.',\n"
    "    tools=[get_weather],\n"
    ")\n"
)

with open(os.path.join(AGENT_DIR, "agent.py"), "w") as f:
    f.write(AGENT_CODE)

# requirements.txt — pinned
with open(os.path.join(AGENT_DIR, "requirements.txt"), "w") as f:
    f.write("google-adk==2.7.1\nlitellm==1.85.7\nopenai==2.45.0\npython-dotenv==1.0.1\ndeprecated==1.2.18\n")

# .env — the key for the running agent
with open(os.path.join(AGENT_DIR, ".env"), "w") as f:
    f.write(f"OPENROUTER_API_KEY={OPENROUTER_API_KEY}\n")

print(f"✅ Deployable layout at {AGENT_DIR}")
print(f"   {sorted(os.listdir(AGENT_DIR))}")

✅ Deployable layout at /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m10_71j2cqpr/my_weather_agent
   ['.env', '__init__.py', 'agent.py', 'requirements.txt']


### 🔍 What just happened?

A complete, deployable agent now exists on disk — and every file in it is something you already know: `agent.py` holds an ordinary `LlmAgent` with one tool, `requirements.txt` pins the same versions our pip cell installs, `.env` carries the key. No new concepts. Deployability is a folder convention, not a technology.

# Run It as a Real HTTP Service

`adk api_server` is the same `adk` command-line tool you met with `adk web` — but instead of opening a chat page, it starts a plain **web server**: your agent behind HTTP, answering JSON requests. That is what a production agent *is*. Cloud Run, later, will be this exact server — just running on Google's machines instead of yours.

The next cell starts it in the background. The comments decode the two new pieces: `subprocess.Popen` ("start another program and let it keep running next to me" — in M02, `McpToolset` did this for you; now you do the starting) and `signal.SIGINT` (Ctrl+C, sent from code, for stopping it politely later).

In [4]:
# Start adk api_server as a background process.
# subprocess.Popen = start another program and keep running (in M02, McpToolset
# did exactly this for you; now you do it yourself). signal.SIGINT is "Ctrl+C,
# sent from code" — we use it at the end to stop the server politely.
import subprocess, signal

api_proc = subprocess.Popen(
    ["adk", "api_server", DEPLOY_DIR, "--port=8765"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env={**os.environ},
)
time.sleep(5)   # give it time to boot
print(f"✅ adk api_server started (PID {api_proc.pid}), hosting {DEPLOY_DIR}")

✅ adk api_server started (PID 18417), hosting /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m10_71j2cqpr


## Now Talk to It — the Way a Customer Would

The server is up. But how do you talk to an agent that is *not* in your notebook anymore? Over HTTP. The next cell uses `urllib.request` — Python's built-in way of making web requests (the same job the OpenAI client does internally; nothing to install).

Watch the shape of the conversation — it mirrors what `chat()` did all course:

1. `GET /list-apps` — *"which agents live here?"*
2. `POST .../sessions/demo-1` — open a session (the same Session concept, now living on the server).
3. `POST /run` — send a user message and get the **event stream** back as JSON.

Same building blocks — Agent, Session, Events — reached through HTTP now.

In [5]:
# Hit it with a real HTTP call. urllib.request is HTTP from the standard
# library — same job as the OpenAI client's internals, no extra install.
import json, urllib.request, urllib.error

# The API mirrors Runner.run_async — create a session, then send messages.
BASE = "http://localhost:8765"

def http_post(path, body):
    req = urllib.request.Request(
        BASE + path,
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode())

# Discover what agents are loaded
with urllib.request.urlopen(BASE + "/list-apps") as r:
    apps = json.loads(r.read().decode())
print(f"Available apps: {apps}")

# Create a session
session = http_post(
    "/apps/my_weather_agent/users/tester/sessions/demo-1",
    {},
)
print(f"Session created: id={session.get('id')}")

# Send a message and collect the final response
run_body = {
    "app_name": "my_weather_agent",
    "user_id": "tester",
    "session_id": "demo-1",
    "new_message": {
        "role": "user",
        "parts": [{"text": "What's the weather in Prague?"}],
    },
}
events = http_post("/run", run_body)
print(f"\nGot {len(events)} event(s) back.")
# Find the final text response
for ev in events:
    if ev.get("content", {}).get("parts"):
        for p in ev["content"]["parts"]:
            if p.get("text"):
                print(f"[{ev.get('author','?')}] {p['text'][:150]}")
            if p.get("function_call"):
                print(f"[tool_call] {p['function_call'].get('name')}")

Available apps: ['my_weather_agent']
Session created: id=demo-1



Got 3 event(s) back.
[weather_agent] Prague is currently cloudy with a temperature of 14°C.


### 🔍 What just happened?

Read the printed lines: a tool call and a final answer — the same events you have read all course, now arriving as JSON over HTTP. And here is the sentence that makes deployment feel small: **swap `http://localhost:8765` for a Cloud Run URL and this exact code keeps working.** The client could be a website, a Slack bot, a phone app — anything that speaks HTTP.

One more cell before moving on: stop the server (that Ctrl+C, sent from code).

In [6]:
api_proc.send_signal(signal.SIGINT)
time.sleep(1)
api_proc.wait(timeout=5)
print("✅ api_server stopped.")

✅ api_server stopped.


### 🎯 Mini-task

In a terminal, copy the agent folder somewhere durable and run `adk api_server <that_dir>` yourself. Then open `http://localhost:8000/docs` — an interactive page listing every endpoint the server offers, generated for free. Try creating a session from that page instead of from code.

# Read-Along: Docker — the Same Server, in a Box

From here to the end of the module **nothing executes for real** — these are read-along cells you will reuse the day you actually deploy. Relax and read.

First path: **Docker**. A container is a box holding your agent, its Python, and its packages — a box any cloud can run. The recipe for building the box is a `Dockerfile`, and ours is five instructions long. The next cell only writes it to disk so you can read it; the comments explain each line. The last line is the whole story: the container simply runs `adk api_server` — the very server you just used.

In [7]:
DOCKERFILE = '''\
FROM python:3.12-slim

WORKDIR /app

# Install Python deps first (separate layer, better caching)
COPY my_weather_agent/requirements.txt ./requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy the agent code
COPY my_weather_agent/ ./my_weather_agent/

# Cloud Run and most PaaS platforms set $PORT; default to 8080 otherwise.
ENV PORT=8080

# Start the ADK API server hosting the agent folder.
# --host 0.0.0.0 so the container is reachable from outside.
CMD ["sh", "-c", "adk api_server /app --host 0.0.0.0 --port ${PORT}"]
'''

DOCKERFILE_PATH = os.path.join(DEPLOY_DIR, "Dockerfile")
with open(DOCKERFILE_PATH, "w") as f:
    f.write(DOCKERFILE)

print("Dockerfile written:")
print(DOCKERFILE)

Dockerfile written:
FROM python:3.12-slim

WORKDIR /app

# Install Python deps first (separate layer, better caching)
COPY my_weather_agent/requirements.txt ./requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy the agent code
COPY my_weather_agent/ ./my_weather_agent/

# Cloud Run and most PaaS platforms set $PORT; default to 8080 otherwise.
ENV PORT=8080

# Start the ADK API server hosting the agent folder.
# --host 0.0.0.0 so the container is reachable from outside.
CMD ["sh", "-c", "adk api_server /app --host 0.0.0.0 --port ${PORT}"]



The build and run commands, in a shell:

```bash
# From the directory containing the Dockerfile
docker build -t my-weather-agent .

# Run locally, passing the API key as an env var
docker run -p 8080:8080 \
    -e OPENROUTER_API_KEY=sk-or-... \
    my-weather-agent

# Hit it
curl http://localhost:8080/list-apps
```

To deploy elsewhere: **AWS Fargate / App Runner** (push to ECR, point a service at the image), **Azure Container Apps** (`az acr build` + `az containerapp create`), **Fly.io** (`fly launch`), **Kubernetes** (your usual manifests).

The property worth remembering: **there is no ADK-specific deployment story for non-Google clouds — and that's good news.** The agent is a normal FastAPI app in a normal container, and those run everywhere.

# Read-Along: One Command for Google Cloud

If your company already lives on Google Cloud, ADK collapses the whole Docker dance into one command:

```bash
adk deploy cloud_run \
    --project YOUR_GCP_PROJECT \
    --region europe-west1 \
    --service_name my-weather-agent \
    my_weather_agent
```

What happens underneath: ADK generates a Dockerfile (like the one you just read), builds the image with Cloud Build, pushes it to Artifact Registry, and deploys a Cloud Run service. All scripted.

| Pro | Con |
|---|---|
| One command, minutes to production | Google Cloud only |
| Sensible defaults for an agent workload | Less control than hand-rolling |
| Plugs into Cloud Trace / Cloud Logging naturally | You pay the Cloud Run runtime margin |

If you're on GCP and in a hurry — use this. If you're on AWS, Azure, on-prem, or you care about the details — the Docker path above is yours.

# Read-Along: Agent Engine — the Managed Path

The third path is **Vertex AI Agent Engine**: you hand over the agent, Google runs it — and makes most of the operations decisions for you. What it adds on top of Cloud Run:

- **Managed sessions** — no database of your own to run.
- **Memory Bank** — long-term memory distilled by an LLM, with automatic consolidation and decay. A genuine upgrade over what we built in M08.
- **Agent Identity** — each agent gets its own IAM identity with certificate-bound credentials; the strongest enterprise-governance feature Google ships.
- **Built-in evaluation and observability** — connects to the same evaluation service M09 used.
- **Framework support**: ADK (native), LangChain, LangGraph, plus custom templates for CrewAI, OpenAI Agents SDK, FastAPI.

Pricing (April 2026 — verify current rates before quoting anyone):

- Runtime: $0.0864/vCPU-hour + $0.0090/GiB-hour (same as Cloud Run; no extra margin on compute).
- Sessions: $0.25 per 1k events.
- Memory Bank: $0.25 per 1k memories stored/month; $0.50 per 1k retrieved (first 1k/month free).

**Choosing between the three paths:**

- **Agent Engine** if you want Memory Bank and Agent Identity without building them — that is the real pitch.
- **Cloud Run** if you want the same container to be able to move elsewhere tomorrow.
- **Neither** if you are not on Google Cloud — the Docker path covers you.

The command, for the day you need it:

```bash
adk deploy agent_engine \
    --project YOUR_PROJECT \
    --region europe-west1 \
    --staging_bucket gs://your-staging-bucket \
    my_weather_agent
```

# Plugins — One Rule for Every Agent

Callbacks (M07) guard *one agent at a time*. Production usually needs the same rule on **every agent in the app**: audit logging, per-user rate limits, token budgets, company-wide PII redaction. Writing the same callback onto twenty agents is exactly the kind of repetition frameworks exist to remove.

**Plugins** are that mechanism. The key move: a plugin is registered **on the `Runner`, not on an agent** — so every agent the runner touches goes through it:

```python
runner = Runner(
    agent=root_agent,
    app_name="my_app",
    session_service=sessions,
    plugins=[AuditPlugin()],   # ← applies to EVERY agent this runner drives
)
```

And a plugin itself is a small class with the same before/after hooks you know from callbacks:

```python
class AuditPlugin(Plugin):
    async def before_run(self, context):
        logger.info("agent_invoked", user=context.user_id, agent=context.agent.name)
    async def after_run(self, context):
        logger.info("agent_finished", duration_ms=context.duration_ms)
```

**Rule of thumb:** callbacks for one agent's specific guards; plugins for rules the whole app must follow (audit, rate limits, cost tracking). Plugins are the newer mechanism and ADK recommends them over app-wide callbacks — if you ever refactor an older ADK deployment, this is the direction to migrate.

# The Production Checklist

Things to have sorted before you deploy — beyond the agent working:

1. **Persistence** — `DatabaseSessionService` on managed Postgres, not in-memory, not SQLite on a disk that vanishes.
2. **Secrets** — `.env` never goes into the image; use Cloud Run secrets, AWS Secrets Manager, or similar.
3. **Callbacks for guardrails** — the blocklist and PII patterns from M07.
4. **Plugins for observability** — audit logging, cost tracking, rate limiting.
5. **Eval in CI** — `adk eval` runs on every change against a curated evalset (M09). Your safety net.
6. **Trace backend** — Cloud Trace, Langfuse, or any OpenTelemetry collector; the events are already instrumented, you just need a receiver.
7. **Health endpoint** — `/list-apps` exists; tell your load balancer what "healthy" means.
8. **Auth** — ADK ships none; put an API gateway in front or run inside an authenticated network.

None of this is ADK-specific — it is what any Python service needs. Worth saying out loud, because the demos give you the agent and everything else is yours to build.

### 🎯 Mini-tasks (for a rainy afternoon)

1. **Build the box.** `docker build -t adk-weather .`, then `docker run -p 8080:8080 -e OPENROUTER_API_KEY=... adk-weather`, then `curl http://localhost:8080/list-apps`.
2. **Deploy for real.** On GCP: run the `adk deploy cloud_run` command. Anywhere else: deploy the container — or just run it locally and point any frontend at it.
3. **Write a plugin.** An `AuditPlugin` that logs user id, agent name and duration for every run. Register it on a Runner and watch the log lines appear.

# Cleanup

Remove the temporary agent folder.

In [8]:
shutil.rmtree(DEPLOY_DIR, ignore_errors=True)
print(f"✅ Cleaned up {DEPLOY_DIR}")

✅ Cleaned up /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m10_71j2cqpr


# Part 1 Wrap — What You Can Build Now

Ten modules, from "what is an agent" to "how do I ship one":

- Agents with tools of four kinds — function, OpenAPI, MCP, agent-as-tool (M02)
- Persistent state with scope prefixes (M03) and long-term memory (M08)
- The one-line model swap across vendors (M04)
- Compositions: Sequential, Parallel, Loop, and LLM-driven routing (M05, M06)
- Guardrails, caches and PII redaction via callbacks (M07)
- Automated evaluation with trajectory checks and LLM-as-judge (M09)
- And now: the same agent as an HTTP service, deployable anywhere

All of it on whichever model you like — because the LiteLlm adapter made the model a *setting*, not a dependency.

# Next up — M11: What Only Gemini Unlocks (Part 2)

Part 2 switches to native Gemini via the `google-genai` SDK: Google Search grounding with real citations, context caching (90% discount on cached tokens), thinking budgets, and the Live voice API. Add a `GOOGLE_API_KEY` to your `.env` (the free tier at aistudio.google.com is enough). See you there.